# 05 — Preference filtering + travel time

Demonstrates `reasoning/preference_filter.py`: "find me a <POI type>, with
<amenities>, ranked by how fast I can actually get there" — combining the
category/amenity data from KG Modelling with the GTFS travel-time reasoning
from `04_reasoning_travel_time.ipynb`.

Design choices (see the module's docstring for the full rationale):
- **Always sorted by travel time ascending** when reachable; POIs with no
  direct connection are still returned, listed after, not silently dropped —
  a hard `max_travel_time_min` cutoff is opt-in only.
- **Fixed bug found while building this**: joining the class-match and
  amenity-match SPARQL patterns in one query triggered a severe rdflib
  query-planning pathology (a query that should take well under a second took
  35+ seconds and had to be killed). Worked around by running the two lookups
  as separate simple queries and intersecting the URI sets in Python — both
  sub-queries alone run in ~0.15-0.25s.

## Setup

In [1]:
import sys
sys.path.insert(0, "..")

from rdflib import Graph
from reasoning.gtfs_routing import GtfsRouter
from reasoning.preference_filter import find_pois

g = Graph()
g.parse("../kg/vienna_mobility_kg.ttl", format="turtle")
router = GtfsRouter(date="20260815")
print(f"KG: {len(g)} triples, GTFS router ready")

KG: 80485 triples, GTFS router ready


## 1. A working example: parks near a museum, ranked by travel time

Same origin as the earlier notebook's worked example (Sammlung alter
Musikinstrumente / Burgring stop), which we already know connects directly to
several parks via tram line 2.

In [2]:
q = """
PREFIX schema: <https://schema.org/>
PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
SELECT ?lon ?lat WHERE {
    ?m a schema:Museum ; schema:name "Sammlung alter Musikinstrumente" ; geo:long ?lon ; geo:lat ?lat .
}
"""
origin_lon, origin_lat = [(float(r.lon), float(r.lat)) for r in g.query(q)][0]

results = find_pois(g, router, origin_lon, origin_lat, poi_classes=["Park"], top_n=15)
n_reachable = sum(1 for r in results if r["reachable_direct"])
print(f"{n_reachable} of {len(results)} parks have a direct connection from here\n")

for r in results:
    status = f"{r['travel_time_min']} min via {r['line']}" if r["reachable_direct"] else "no direct route"
    print(f"  {r['name']:30} {status}")

5 of 15 parks have a direct connection from here

  PA Beethovenplatz              12.3 min via 2
  Manes-Sperber-Park             21.6 min via 2
  GA Adele-Perlmutter-Platz      22.3 min via 2
  PA Dresdner Straße             32.3 min via 2
  PA Meldemannstraße             33.0 min via 2
  Fritz-Imhoff-Park              no direct route
  GA St.-Wendelin-Platz          no direct route
  Herklotzpark                   no direct route
  PA Chromygasse                 no direct route
  Ferdinand-Wolf-Park            no direct route
  PA Bodenstedtgasse             no direct route
  PA Schuhmeierplatz             no direct route
  Adelheid-Popp-Park             no direct route
  PA Colerusgasse                no direct route
  Donauschwabenpark              no direct route


## 2. A hard time cutoff

Same query, but only results reachable within 25 minutes — the earlier
32-33 min results should drop out.

In [3]:
results_25 = find_pois(g, router, origin_lon, origin_lat, poi_classes=["Park"],
                        max_travel_time_min=25, top_n=15)
print(f"{len(results_25)} parks within 25 min:")
for r in results_25:
    print(f"  {r['name']:30} {r['travel_time_min']} min via {r['line']}")

3 parks within 25 min:
  PA Beethovenplatz              12.3 min via 2
  Manes-Sperber-Park             21.6 min via 2
  GA Adele-Perlmutter-Platz      22.3 min via 2


## 3. Combining a category with a required amenity — and an honest empty result

"Dog-friendly park" adds a `schema:amenityFeature` requirement on top of the
class filter. Worth showing even though (spoiler) it comes back empty from
this particular origin — that's a real, expected outcome given how rare direct
connections are (well under 1% of pairs, per `04_reasoning_travel_time.ipynb`),
not a bug. A preference filter that only ever returns matches when you get
lucky with the origin wouldn't be honest about that limitation.

In [4]:
dog_parks = find_pois(g, router, origin_lon, origin_lat, poi_classes=["Park"],
                       required_amenities=["Dogs allowed"], top_n=10)
n_reachable = sum(1 for r in dog_parks if r["reachable_direct"])
print(f"Dog-friendly parks: {n_reachable} of {len(dog_parks)} reachable directly from this origin")
print("(all results still returned and listed, just none marked reachable -- see below)\n")
for r in dog_parks[:5]:
    print(f"  {r['name']:30} {'no direct route' if not r['reachable_direct'] else r['travel_time_min']}")

Dog-friendly parks: 0 of 10 reachable directly from this origin
(all results still returned and listed, just none marked reachable -- see below)

  PA Lautenschlägergasse         no direct route
  PA Flammweg                    no direct route
  Erika-Morini-Park              no direct route
  Claudia-Heill-Park             no direct route
  Edelsinnstraße Hundezone       no direct route


## 4. A different amenity: playground equipment

Equipment-list amenities (parsed from `SPIELPLATZ_DETAIL`) work the same way
as Park's boolean amenities — same `schema:amenityFeature` pattern, per the
KG Modelling decision to use one mechanism for both.

In [5]:
trampoline_playgrounds = find_pois(g, router, origin_lon, origin_lat,
                                    poi_classes=["PlaygroundArea"],
                                    required_amenities=["Trampolin"], top_n=10)
print(f"Playgrounds with a trampoline: {len(trampoline_playgrounds)} found")
for r in trampoline_playgrounds:
    status = f"{r['travel_time_min']} min via {r['line']}" if r["reachable_direct"] else "no direct route"
    print(f"  {r['name']:30} {status}")

Playgrounds with a trampoline: 10 found
  GA Franklinstraße              no direct route
  Alfred-Böhm-Park               no direct route
  PA Wielandplatz                no direct route
  Laubepark                      no direct route
  Türkenschanzpark               no direct route
  PA Carminweg                   no direct route
  Andreaspark                    no direct route
  Donaupark                      no direct route
  Humboldtpark                   no direct route
  Vally-Wieselthier-Park         no direct route


## Findings

**The core mechanism works**: category filtering, amenity matching, and
travel-time ranking all combine correctly, reusing exactly the same
`GtfsRouter` and `schema:amenityFeature` pattern built in earlier chapters —
no new KG modelling needed for this reasoning capability.

**A real annoying bug was in query construction, not the data or the design per se.**
Worth remembering for any future SPARQL work on this graph: joining a
class-match pattern with an amenity-match pattern (plus a `CONTAINS`/`LCASE`
filter) in one query is drastically slower than running them separately and
intersecting in Python, at least on rdflib's default query engine. Not an
obvious thing to predict in advance — worth testing incrementally rather than
assuming a "natural-looking" SPARQL query will perform well.

**Direct-connection rarity shows up here too, compounded.** Requiring *both*
a specific amenity *and* a direct connection narrows things fast — the
dog-friendly-park example came back with zero reachable results from this
particular origin. That's consistent with earlier findings, not a new
problem, but it's a real UX question for the eventual Service Layer: what
should happen when a preference query has no directly-reachable matches?
Options include falling back to walking-distance-only suggestions, relaxing
the "direct connection only" constraint for that query, or just being upfront
that nothing matched. Not decided yet.